# U-RNN Quick Start — Urban Flood Nowcasting

[![Journal of Hydrology 2025](https://img.shields.io/badge/Journal%20of%20Hydrology-2025-blue)](https://doi.org/10.1016/j.jhydrol.2025.133117)
[![GitHub](https://img.shields.io/badge/GitHub-holmescao/U--RNN-black?logo=github)](https://github.com/holmescao/U-RNN)

This notebook demonstrates **U-RNN inference** using pre-trained weights on the UrbanFlood24 dataset.

**What you'll see in ~5 minutes:**
- Load a pre-trained U-RNN (location1, 1000 epochs)
- Run inference on a 100-year return-period rainfall event
- Visualise a 3-row comparison: Reference | U-RNN Prediction | Absolute Error

**Runtime:** Free Colab T4 GPU (~5 min total)

---
> **Paper:** Cao et al. (2025) *U-RNN high-resolution spatiotemporal nowcasting of urban flooding*, Journal of Hydrology.  
> DOI: [10.1016/j.jhydrol.2025.133117](https://doi.org/10.1016/j.jhydrol.2025.133117)

## Step 1 — Environment Setup

Install PyTorch and project dependencies. This takes ~2 minutes on a fresh Colab runtime.

In [ ]:
# Install PyTorch (Colab usually has a compatible version pre-installed)
import torch
print(f'PyTorch: {torch.__version__}, CUDA available: {torch.cuda.is_available()}')

# Clone the repository
!git clone https://github.com/holmescao/U-RNN /content/U-RNN --quiet
%cd /content/U-RNN/U-RNN

# Install core dependencies
!pip install -r requirements.txt -q

## Step 2 — Download Pre-trained Weights & Sample Data

We download:
- Pre-trained checkpoint (location1, epoch 939)
- One test event from UrbanFlood24 (r100y, 100-year storm)

> Files are hosted on Google Drive. If the download fails, see the [README](https://github.com/holmescao/U-RNN#3-pre-trained-weights) for Baidu Cloud links.

In [ ]:
import os

# ── Pre-trained weights ──────────────────────────────────────────────────────
CHECKPOINT_DIR = '/content/U-RNN/exp/20240202_162801_962166/save_model'
CHECKPOINT_FILE = 'checkpoint_939_0.000205376.pth.tar'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Download checkpoint from Google Drive
# NOTE: Replace FILE_ID below if the link changes
GDRIVE_FILE_ID = '1tfwRJ3gFFTa0kiziVeo9xXsz0DaaJrJU'
!gdown --id {GDRIVE_FILE_ID} -O {CHECKPOINT_DIR}/{CHECKPOINT_FILE} --quiet
print(f'Checkpoint: {os.path.getsize(os.path.join(CHECKPOINT_DIR, CHECKPOINT_FILE)) / 1e6:.1f} MB')

# ── Sample data ──────────────────────────────────────────────────────────────
# TODO: replace DATA_GDRIVE_ID with the ID of a single-event sample zip
# hosted on the project's Google Drive once the sample dataset is uploaded.
DATA_GDRIVE_ID = 'YOUR_SAMPLE_DATA_GDRIVE_ID'
DATA_ROOT = '/content/U-RNN/data/urbanflood24'
os.makedirs(DATA_ROOT, exist_ok=True)

print('\nNote: Replace DATA_GDRIVE_ID above with the sample dataset file ID.')
print('See https://github.com/holmescao/U-RNN#2-dataset-preparation for download links.')

## Step 3 — Run Inference

Run `test.py` to generate predictions and the 3-row comparison figure.

In [ ]:
%cd /content/U-RNN/U-RNN

!python test.py \
    --exp_config configs/full.yaml \
    --timestamp  20240202_162801_962166 \
    --device     0

## Step 4 — Visualise Results

Display the Reference / U-RNN / Error comparison figure for the first test event.

In [ ]:
import glob
from IPython.display import Image, display

figs = sorted(glob.glob(
    '/content/U-RNN/exp/20240202_162801_962166/figs/epoch@*/*/water_depth_spatial_temporal.png'
))

if figs:
    print(f'Found {len(figs)} result figure(s). Displaying the first:')
    for f in figs[:2]:
        print(f'  {f}')
        display(Image(filename=f, width=900))
else:
    print('No figures found — check that inference completed successfully.')

## Step 5 — Train on the Lightweight Dataset (Optional)

Train U-RNN from scratch on the downsampled dataset (125×125×36). Takes ~30 min on a T4 GPU.

In [ ]:
# First downsample the dataset:
!python tools/downsample_dataset.py \
    --src_root  ../data/urbanflood24 \
    --dst_root  ../data/urbanflood24_lite \
    --spatial_factor  4 \
    --temporal_factor 10

# Then train (200 epochs, ~30 min):
!python main.py --exp_config configs/lite.yaml --device 0

## Next Steps

| Goal | How |
|---|---|
| Train on your own catchment | See [README Section 13 FAQ](https://github.com/holmescao/U-RNN#13-faq) — "What do I need to change when using my own dataset?" |
| Train on Futian (China) or UKEA (UK) | Use `configs/futian_scratch.yaml` or `configs/ukea_scratch.yaml` |
| Transfer learning | Load Futian weights → fine-tune to new location in 30 epochs via `configs/finetune_location1_from_futian.yaml` |
| Speed up inference | Convert to TensorRT — see [README Section 8](https://github.com/holmescao/U-RNN#8-inference-with-tensorrt-optional) |
| Report a bug | [Open an issue](https://github.com/holmescao/U-RNN/issues/new/choose) |

**If this notebook was useful, please ⭐ the repository!**

[![GitHub stars](https://img.shields.io/github/stars/holmescao/U-RNN?style=social)](https://github.com/holmescao/U-RNN)